# 🤖 在 Colab 執行小組 LINE Bot

這份筆記本會從 GitHub 下載你們組的 repo，然後用和 W06 一樣的 Dev Tunnel 方式啟動 Bot。

**每次上課的流程：** 步驟 0 → 1 → 2（不用 LINE 也能測）→ 3 → 4 → 5。
合併新的 PR 之後，只要**停止步驟 5**，重跑**步驟 0 和 5**，Bot 就會換成最新版。

> 🔐 Channel secret / access token 只用 `getpass` 輸入，**不要**寫進任何一格程式碼，也不要 commit 到 GitHub。

In [ ]:
#@title 步驟 0：下載（或更新）你們組的程式碼
#@markdown 把網址換成你們組的 repo，例如 `https://github.com/組織名稱/G1`
REPO_URL = "https://github.com/your-org/G1"  #@param {type:"string"}

import os, sys
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C "$REPO_DIR" pull --ff-only
else:
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("\n📜 最近 5 個 commit：")
!git log --oneline -5

In [ ]:
#@title 步驟 1：安裝套件並跑測試
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
#@title 步驟 2：不用 LINE 也能測試 🧪
#@markdown 輸入一句話，看看 Bot 會怎麼回。輸入「功能」看所有功能。
TEST_TEXT = "功能"  #@param {type:"string"}

import importlib
import router
importlib.reload(router)
modules = router.load_features(reload=True)
print(f"已載入 {len(modules)} 個功能：", ", ".join(m.NAME for m in modules))
print("你>", TEST_TEXT)
for msg in router.route(TEST_TEXT, "colab-test", modules):
    print("bot>", getattr(msg, "text", None) or f"[{type(msg).__name__}]")

# 3 設定對外通道（和 W06 相同）

In [ ]:
#@title 3-1 安裝 Microsoft Dev Tunnel CLI
!curl -sL -o /usr/bin/devtunnel https://tunnelsassetsprod.blob.core.windows.net/cli/linux-x64-devtunnel
!sudo apt-get -qq update -y
!sudo apt-get -qq install -y libsecret-1-0
!chmod +x /usr/bin/devtunnel
!devtunnel --version

In [ ]:
#@title 3-2 連結 GitHub 帳號
#@markdown 點選連結，使用 GitHub 帳戶登入，並輸入顯示的 8 碼
!devtunnel user login -g -d

In [ ]:
#@title 3-3 公開 port 5000（背景執行）
!nohup devtunnel host -p 5000 --allow-anonymous > /content/devtunnel_out.txt 2>&1 &

In [ ]:
#@title 3-4 顯示公開網址，設定 LINE Webhook URL
#@markdown 如果沒有公開網址，請再執行一次本步驟
public_url = []
with open('/content/devtunnel_out.txt', 'r') as f:
    for one_line_text in f.readlines():
        if "Connect via browser" in one_line_text:
            public_url = [url.strip() for url in ":".join(one_line_text.split(':')[1:]).split(',')]
if public_url:
    for i in public_url:
        print(i)
    print("👉 選其中一個網址，貼到 LINE Developers 的 Webhook URL")
else:
    print("如果沒有公開網址，請再執行一次本步驟")

# 4 輸入 LINE 授權令牌

In [ ]:
#@title 4 輸入 Channel secret 與 access token（不會顯示、不會存檔）
import getpass
print("貼上你的 LINE Channel secret: (https://developers.line.biz/console/)")
secret = getpass.getpass()
print("貼上你的 LINE Channel access token: (https://developers.line.biz/console/)")
access_token = getpass.getpass()

# 5 啟動 Bot
這一格會一直執行（這是正常的）。要更新程式碼時：按 ⏹ 停止 → 重跑步驟 0 → 再跑這一格。

In [ ]:
#@title 5 啟動小組 LINE Bot（port 5000）
import importlib
import router, app
importlib.reload(router)
importlib.reload(app)

for i in globals().get("public_url", []):
    print("🌐", i)
app.create_app(secret, access_token).run(port=5000)

# 6 關閉通道（下課前執行）

In [ ]:
#@title 6-1 關閉所有 devtunnel 程序
!ps -ef | grep devtunnel > /content/processes_out.txt

import os, signal
with open('/content/processes_out.txt', 'r') as f:
    for one_line_text in f.readlines():
        columns = [text for text in one_line_text.split(' ') if text]
        try:
            os.kill(int(columns[1]), signal.SIGTERM)
        except Exception:
            pass
print("已關閉 devtunnel")

In [ ]:
#@title 6-2 登出 devtunnel
!devtunnel user logout